## Preparando os dados via API

In [0]:
!pip install ccxt

In [0]:
import ccxt

par = 'BTC/USDT'
limit = 5000
timeframe = '1h' # 15m / 4h / 1h

# conexão com a API. enableRateLimit para não exceder o limite de requisições
# utilizando Binance US pois o outro está bloqueando meu IP. talvez porque o servidor do databricks fique nos EUA
binance = ccxt.binanceus({'enableRateLimit': True})
print(binance)

# limit = quantidade de dados (candles) que queremos retornar
ohlcv = binance.fetch_ohlcv(par, timeframe=timeframe, limit=limit)
# ohlcv

In [0]:
((limit*60) / 60) / 24

**ccxt:** CryptoCurrency eXchange Trading - conectar e negociar em diversas exchanges de criptomoedas

**Alguns significados**
- O (Open / Abertura): O preço do ativo no exato momento em que o intervalo de tempo começou (ex: o preço às 10:00:00)
- H (High / Máxima): O preço mais alto que o ativo atingiu durante aquele intervalo
- L (Low / Mínima): O preço mais baixo que o ativo atingiu durante aquele intervalo
- C (Close / Fechamento): O preço do ativo no momento em que o intervalo terminou (ex: o preço às 10:59:59)
- V (Volume): A quantidade total do ativo que foi negociada (comprada/vendida) naquele período

In [0]:
import pyspark.sql.types as t

# criando dataframe
schema = t.StructType([
    t.StructField("timestamp"   ,t.LongType()     ,True)
    ,t.StructField("open"       ,t.DoubleType()   ,True)
    ,t.StructField("high"       ,t.DoubleType()   ,True)
    ,t.StructField("low"        ,t.DoubleType()   ,True)
    ,t.StructField("close"      ,t.DoubleType()   ,True)
    ,t.StructField("volume"     ,t.DoubleType()   ,True)
])
df_ohlcv = spark.createDataFrame(ohlcv, schema)

# convertendo a informação de data para timestamp
df_ohlcv = df_ohlcv.withColumn("timestamp", (df_ohlcv.timestamp / 1000).cast("timestamp"))
# df_ohlcv.display()

 Não está no nosso fuso horário

In [0]:
from pyspark.sql.functions import from_utc_timestamp
df_ohlcv = df_ohlcv.withColumn("timestamp", from_utc_timestamp("timestamp", "America/Sao_Paulo"))

In [0]:
close_s     = df_ohlcv.select("close")
high_s      = df_ohlcv.select("high")
low_s       = df_ohlcv.select("low")
volume_s    = df_ohlcv.select("volume")

## Feature engineering

In [0]:
from pyspark.sql.window import Window
import pyspark.sql.functions as f

w = Window.orderBy("timestamp")

#### Log return (retorno logarítmico)
$$r_t = \ln(P_t) - \ln(P_{t-1})$$
- Cálculo do retorno de um ativo entre dois períodos usando logaritmos naturais
- Permite somar retornos de diferentes períodos
- adequado para séries temporais

In [0]:
df_ohlcv.display()

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

No dia 26 de março às 14:45, houve um pico de volume bem grande, porém, não mexeu nos preços seguintes. seria um sinal de exaustão?

In [0]:
df_feat = df_ohlcv.withColumn(
    "log_return",
    f.log(f.col("close") / f.lag(f.col("close"), 1).over(w))
)

#### RSI (Índice de Força Relativa)
$$RSI = 100 - \left( \frac{100}{1 + RS} \right)$$
Onde o RS (Força Relativa) é:
$$RS = \frac{\text{media dos ganhos}}{\text{media das perdas}}$$

- Mede a velocidade e a mudança dos movimentos de preço
- Indica condições de sobrecompra (>70) ou sobrevenda (<30)
- Utilizado para identificar possíveis reversões de tendência
- Geralmente, utiliza-se um período de 14 candles para esse cálculo

In [0]:
periodo = 14
periodo = periodo - 1

delta = f.col("close") - f.lag(f.col("close"), 1).over(w)

ganho = f.when(delta > 0, delta).otherwise(0)
perda = f.when(delta < 0, -delta).otherwise(0)

avg_ganho = f.avg(ganho).over(Window.orderBy("timestamp").rowsBetween(-13, 0))
avg_perda = f.avg(perda).over(Window.orderBy("timestamp").rowsBetween(-13, 0))

rs = avg_ganho / avg_perda
rsi = f.when(avg_perda == 0, 100).otherwise(100 - (100 / (1 + rs)))

df_feat = df_feat.withColumn("rsi", rsi)

### Variação de volume
- mede a alteração na intensidade das negociações entre dois períodos
- valida se um movimento de preço tem relevância ou se é apenas um ruído de baixa liquidez
$$V_{change} = \frac{V_t - V_{t-1}}{V_{t-1}}$$
- Vt: Volume do período atual
- Vt-1: Volume do período anterior

In [0]:
df_feat = df_feat.withColumn(
    "vol_change"
    ,(f.col("volume") - f.lag(f.col("volume"), 1).over(w)) / f.lag(f.col("volume"), 1).over(w)
)

### Médias móveis, desvio padrão móvel e z-score (janela de 20 períodos)

- **Média móvel:** Suaviza as flutuações de preço, mostrando a tendência geral
$$MA_{20} = \frac{1}{20} \sum_{i=0}^{19} P_{t-i}$$

- **Desvio padrão móvel:** Mede a volatilidade dos preços em uma janela de 20 períodos
$$STD_{20} = \sqrt{\frac{1}{20} \sum_{i=0}^{19} (P_{t-i} - MA_{20})^2}$$

- **Z-score:** Indica o quão distante o preço está da média móvel, em termos de desvio padrão
$$Z_{t} = \frac{P_t - MA_{20}}{STD_{20}}$$

- Pt: Preço do período atual
- MA20: Média móvel dos últimos 20 períodos
- STD20: Desvio padrão dos últimos 20 períodos

In [0]:
periodo = 20
w_movel = Window.orderBy("timestamp").rowsBetween(-periodo + 1, 0)

# Média móvel e Desvio padrão móvel do volume
df_feat = (
    df_feat
    # volume
    .withColumn("media_movel_vol", f.avg("volume").over(w_movel))
    .withColumn("media_movel_close", f.avg("close").over(w_movel))
    # desvio padrão
    .withColumn("std_movel_vol", f.stddev("volume").over(w_movel))
    .withColumn("std_movel_close", f.stddev("close").over(w_movel))
) 

In [0]:
df_feat = (
    df_feat
    # volume
    .withColumn("z_score_vol", (f.col("volume") - f.col("media_movel_vol")) / f.col("std_movel_vol"))
    # fechamento
    .withColumn("z_score_close", (f.col("close") - f.col("media_movel_close")) / f.col("std_movel_close"))
) 

In [0]:
# limite de anomalia (Z-Score > 3)
df_feat = df_feat.withColumn("vol_anomalia", f.col("z_score_vol") > 3)

# sinal que combina preço + anomalia de volume
df_feat = df_feat.withColumn(
    "vol_signal",
    f.when(
        (f.col("vol_anomalia")) & (f.col("close") > f.col("open")),
        "Forte compra (baleia)"
    ).when(
        (f.col("vol_anomalia")) & (f.col("close") < f.col("open")),
        "Forte venda (dump)"
    ).otherwise("Normal")
)

### Assimetria e Curtose (janela móvel)
- **Assimetria:** Mede a simetria da distribuição dos preços em relação à média. Valores positivos indicam cauda à direita, negativos à esquerda.
- **Curtose:** Mede o grau de "achatamento" da distribuição dos preços. Curtose alta indica mais extremos (caudas pesadas).
$$\text{Assimetria}_t = \text{skewness}(P_{t-19}, ..., P_t)$$
$$\text{Curtose}_t = \text{kurtosis}(P_{t-19}, ..., P_t)$$
- Calculadas sobre a janela móvel de 20 períodos usando a coluna de fechamento (`close`)

In [0]:
# assimetria e curtose
df_feat = df_feat.withColumn("assimetria", f.skewness("close").over(w_movel))
df_feat = df_feat.withColumn("curtosi", f.kurtosis("close").over(w_movel))

### Preço Típico
- preço médio ponderado de um candle
- utilizado em indicadores como o VWAP e alguns osciladores
$$\text{Typical Price}_t = \frac{H_t + L_t + C_t}{3}$$
- Ht: Máxima do período
- Lt: Mínima do período
- Ct: Fechamento do período

### VP
- VP: Valor negociado ponderado pelo preço típico
- Calculado como: VP = Typical_Price * volume

In [0]:
df_feat = (
  df_feat
  # Typical price
  .withColumn("typical_price", (f.col("high") + f.col("low") + f.col("close")) / 3)
  # VP
  .withColumn("VP", f.col("Typical_Price") * f.col("volume"))
) 

### VWAP (Volume Weighted Average Price)
- preço médio ponderado pelo volume negociado em um período
- utilizado para avaliar o preço médio de execução de grandes ordens
$$\text{VWAP}_\text{daily} = \frac{\sum_{i=1}^{N} (\text{Typical Price}_i \times \text{Volume}_i)}{\sum_{i=1}^{N} \text{Volume}_i}$$
- N: número de candles no dia
- Typical Price: preço típico do candle
- Volume: volume negociado no candle

### Distância ao VWAP
- Mede o quanto o preço de fechamento está distante do VWAP diário
$$\text{Dist VWAP}_t = \frac{C_t - \text{VWAP}_\text{daily}}{\text{VWAP}_\text{daily}}$$
- Ct: Fechamento do período
- VWAP_daily: VWAP do dia

In [0]:
w_daily = Window.partitionBy(f.to_date("timestamp")).orderBy("timestamp").rowsBetween(Window.unboundedPreceding, 0)

# Daily VWAP
df_feat =(
  df_feat
  .withColumn("date", f.to_date("timestamp"))
  .withColumn("cum_VP", f.sum("VP").over(w_daily))
  .withColumn("cum_Volume", f.sum("volume").over(w_daily))
  .withColumn("daily_VWAP", f.col("cum_VP") / f.col("cum_Volume"))
)

# Distância ao VWAP
df_feat = df_feat.withColumn("dist_VWAP", (f.col("close") - f.col("daily_VWAP")) / f.col("daily_VWAP"))

# drop columns
df_feat = df_feat.drop("Typical_Price", "VP", "date", "cum_VP", "cum_Volume")
# df_feat.display()

In [0]:
df_feat.limit(10).display()

In [0]:
%skip
(
    df_feat
    .select(
        'timestamp'
        ,'open'
        ,'high'
        ,'low'
        ,'close'
        ,'volume'
    )
    .write
    .format("delta")
    .mode("overwrite")
    .option('overwriteSchema', 'true')
    .saveAsTable("default.tbtc_features")
)

## Modelagem
### Lógica do Escalonamento Proporcional
Podemos usar o tempo em minutos de cada timeframe como a base do nosso cálculo. A ideia é que o threshold cresça com a raiz quadrada do tempo (já que a volatilidade geralmente escala com raiz de t e o horizonte acompanhe a duração da tendência esperada.

In [0]:
# horizonte de previsão (3 horas para frente? depende do timeframe?)
# target = 1 se o preço subir > x% em xh, 0 caso contrário

# # target para cada timeframe
# if timeframe == '15m':
#     # Em 15m, o BTC oscila rápido. Aumentamos o horizonte para dar tempo da tendência maturar.
#     horizonte = 8        # 2 horas para frente (8 * 15m)
#     threshold = 0.004    # 0.4% (Mais realista para cobrir taxas e dar sinal)
    
# elif timeframe == '1h':
#     # O seu anterior de 1% em 3h estava muito rígido (gerou apenas 97 acertos).
#     # Vamos dar mais tempo (6h) e baixar levemente o alvo para 0.8%.
#     horizonte = 6        # 6 horas para frente
#     threshold = 0.008    # 0.8% de ganho
    
# elif timeframe == '4h':
#     # Para 4h, o movimento precisa ser direcional. 
#     horizonte = 3        # 12 horas para frente (3 * 4h)
#     threshold = 0.025    # 2.5% (3% era muito raro no curto prazo)
    
# else:
#     horizonte = 4
#     threshold = 0.005

# 1. Definindo a Janela para Volatilidade (ATR)
w_atr = Window.orderBy("timestamp").rowsBetween(-13, 0)

# 2. Cálculo do True Range (TR)
df_feat = df_feat.withColumn("h_l", f.col("high") - f.col("low"))
df_feat = df_feat.withColumn("h_pc", f.abs(f.col("high") - f.lag("close", 1).over(w)))
df_feat = df_feat.withColumn("l_pc", f.abs(f.col("low") - f.lag("close", 1).over(w)))

# O True Range é o maior desses 3 valores
df_feat = df_feat.withColumn("TR", f.greatest("h_l", "h_pc", "l_pc"))

# 3. Cálculo do ATR (Média do TR)
df_feat = df_feat.withColumn("ATR", f.avg("TR").over(w_atr))

# 1. Definir a base de tempo em minutos para cada timeframe
tf_map = {'15m': 15, '1h': 60, '4h': 240}
minutos_base = tf_map.get(timeframe, 60)

# 2. Definir o "Peso" ou Intensidade (Ajuste apenas aqui!)
# Peso 1.0 = Conservador | Peso 1.5 = Agressivo | Peso 0.8 = Scalper
peso_agressividade = 1.0 

# O Horizonte continua fixo em tempo real (~6 horas), mas muda em candles
horizonte = int(360 / minutos_base)

# O Threshold agora é 1.5x a volatilidade média (ATR) expressa em %
# Isso significa que buscamos um movimento que seja 1.5 vezes o "tamanho comum" do candle
df_feat = df_feat.withColumn(
    "dynamic_threshold", 
    (f.col("ATR") * 1.5) / f.col("close")
)

print(f"--- Configuração Dinâmica para {timeframe} ---")
print(f"Horizonte: {horizonte} candles | Alvo: {threshold*100:.2f}%")

# criando a coluna target
# o 'f.lead' olha para o fechamento futuro e compara com o atual
# Criando o Target com o Threshold Dinâmico
df_model = df_feat.withColumn(
    "target",
    f.when(
        (f.lead("close", horizonte).over(w) > f.col("close") * (1 + f.col("dynamic_threshold"))), 
        1
    ).otherwise(0)
)

In [0]:
# remove nulos gerados pelas janelas e pelo lead do target
df_model = df_model.dropna()

# converte o sinal categórico para numérico
df_model = df_model.withColumn(
    "vol_signal_idx", 
    f.when(f.col("vol_signal") == "Normal", 0)
    .when(f.col("vol_signal") == "Forte compra (baleia)", 1)
    .otherwise(2) # Forte venda (dump)
)

In [0]:
df_model.groupBy("target").count().show()

| target | count |
|--------|-------|
|   0    |  902  |
|   1    |   97  |

as classes estão desbalanceadas
- threshhold está muito alto?

In [0]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml import Pipeline

from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

In [0]:
# Momentum de 3 períodos (Variação da variação)
df_model = df_model.withColumn("momentum", (f.col("close") - f.lag("close", 3).over(w)) / f.lag("close", 3).over(w))
# Hora do dia (Feature Categórica)
df_model = df_model.withColumn("hour", f.hour("timestamp"))
# lista de features para o modelo

features_cols = [
    "log_return", "rsi", "vol_change", "z_score_vol", 
    "z_score_close", "assimetria", "curtosi", "dist_VWAP"
    # , "vol_signal_idx"
    ,"momentum", "hour"
]

df_final = df_model.select(*features_cols, "target", "timestamp").dropna().orderBy("timestamp")

# assembler (Transforma colunas em um único vetor 'features')
assembler = VectorAssembler(inputCols=features_cols, outputCol="features")
df_vector = assembler.transform(df_final)

In [0]:
# ordenando e dividindo
total = df_final.count()
train_size = int(total * 0.8)

train_df = df_final.limit(train_size)
test_df = df_final.subtract(train_df) # o restante é teste

In [0]:
# Split (Treino: 80% / Teste: 20%)
total_rows = df_vector.count()
split_point = int(total_rows * 0.8)

# criando dfs de Treino (passado) e Teste (futuro Recente)
train_df = df_vector.limit(split_point)
test_df = df_vector.subtract(train_df)

In [0]:
# 1. Equilibrando a base (Undersampling)
df_ones = train_df.filter(f.col("target") == 1)
df_zeros = train_df.filter(f.col("target") == 0)

# Deixando a proporção 50/50 ou próxima disso
ratio = df_ones.count() / df_zeros.count()
df_zeros_sampled = df_zeros.sample(withReplacement=False, fraction=ratio, seed=56)
train_df = df_ones.union(df_zeros_sampled)

In [0]:
train_df.count()

In [0]:
# Configuração e Treinamento do Modelo
# Usaremos 100 árvores para um bom equilíbrio entre performance e generalização
# rf = RandomForestClassifier(labelCol="target", featuresCol="features", numTrees=100, seed=42)
# Ajustando para um modelo mais Robusto (menos propenso a decorar ruído)
rf = RandomForestClassifier(
    labelCol="target", 
    featuresCol="features", 
    numTrees=200,          # Mais árvores para estabilizar o sinal
    maxDepth=5,            # Árvores rasas para evitar Overfitting em base pequena
    subsamplingRate=0.7,   # Cada árvore vê apenas 70% dos dados
    featureSubsetStrategy="sqrt",
    seed=42
)
model = rf.fit(train_df)

# Fazendo as previsões
predictions = model.transform(test_df)

# Avaliação das métricas
evaluator_roc = BinaryClassificationEvaluator(labelCol="target", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="target", predictionCol="prediction", metricName="accuracy")

auc = evaluator_roc.evaluate(predictions)
accuracy = evaluator_acc.evaluate(predictions)

print(f"Área sob a Curva ROC (AUC): {auc:.4f}")
print(f"Acurácia Geral: {accuracy:.4f}")

# previsões
predictions.select("timestamp", "target", "prediction", "probability").show(10)

In [0]:
predictions.count()

- att0: Apesar da acurácia estar aparentemente boa (81%), a curva ROC está dizendo que o modelo não está conseguindo distinguir entre as classes, com um AUC de 50% — equivalente a uma escolha aleatória (como jogar uma moeda para decidir se o mercado vai subir ou descer)

- att1: acuracia abaixou para 49% após undersampling, porém, roc abaixou tambem! (44%)

- **att2: acurácia continua 49%, porém curva roc foi para 55%!!! isso é muito bom! ocorreu após utilizar o threshhold dinâmico**

In [0]:
import pandas as pd
# extraindo a importância
importances = model.featureImportances
feat_imp = pd.DataFrame(list(zip(features_cols, importances.toArray())), 
                        columns=['Feature', 'Importance']).sort_values(by='Importance', ascending=False)
print(feat_imp)

as principais features são a Assimetria e o Z-Score do preço. até que faz sentido, pois elas indicam quando o preço está "esticado" ou "desequilibrado"